<a href="https://colab.research.google.com/github/himanshubhimte69/AgriLite-A-Lightweight-Multi-Crop-Disease-Detector-Across-Diverse-Conditions/blob/main/PlantDoc_EfficientNetB2_FineTune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/pratikkayal/PlantDoc-Dataset.git
!mv PlantDoc-Dataset /content/PlantDoc
!ls /content/PlantDoc

Cloning into 'PlantDoc-Dataset'...
remote: Enumerating objects: 2670, done.
remote: Counting objects: 100% (35/35), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 2670 (delta 22), reused 22 (delta 22), pack-reused 2635 (from 1)
Receiving objects: 100% (2670/2670), 932.92 MiB | 13.69 MiB/s, done.
Resolving deltas: 100% (24/24), done.
Updating files: 100% (2581/2581), done.
LICENSE.txt  PlantDoc_Examples.png  README.md  test  train


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.applications import EfficientNetB2, efficientnet_v2
from tensorflow.keras.preprocessing import image_dataset_from_directory
import numpy as np
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, cohen_kappa_score
)
from sklearn.preprocessing import label_binarize
import random

In [ ]:
# ============================
# 1️⃣ Load Dataset
# ============================
img_size = (260, 260)  # EfficientNetB2 default resolution
batch_size = 16

train_ds_full = image_dataset_from_directory(
    "/content/PlantDoc/train",
    image_size=img_size,
    batch_size=batch_size
)

test_ds = image_dataset_from_directory(
    "/content/PlantDoc/test",
    image_size=img_size,
    batch_size=batch_size
)

class_names = train_ds_full.class_names
num_classes = len(class_names)

# Split train → train + validation
val_batches = tf.data.experimental.cardinality(train_ds_full) // 5
val_ds = train_ds_full.take(val_batches)
train_ds = train_ds_full.skip(val_batches)

print("Classes:", class_names)
print("Num Classes:", num_classes)

Found 2342 files belonging to 28 classes.
Found 236 files belonging to 27 classes.
Classes: ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf', 'Bell_pepper leaf spot', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight', 'Potato leaf late blight', 'Raspberry leaf', 'Soyabean leaf', 'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato Early blight leaf', 'Tomato Septoria leaf spot', 'Tomato leaf', 'Tomato leaf bacterial spot', 'Tomato leaf late blight', 'Tomato leaf mosaic virus', 'Tomato leaf yellow virus', 'Tomato mold leaf', 'Tomato two spotted spider mites leaf', 'grape leaf', 'grape leaf black rot']
Num Classes: 28


In [ ]:
# ============================
# 2️⃣ Preprocessing + Augmentation
# ============================
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
])

preprocess_input = efficientnet_v2.preprocess_input

train_ds = train_ds.map(lambda x, y: (preprocess_input(data_augmentation(x)), y))
val_ds   = val_ds.map(lambda x, y: (preprocess_input(x), y))
test_ds  = test_ds.map(lambda x, y: (preprocess_input(x), y))

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds   = val_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

# One-hot encoding
def one_hot_map(images, labels):
    return images, tf.one_hot(labels, depth=num_classes)

train_ds_onehot = train_ds.map(one_hot_map)
val_ds_onehot   = val_ds.map(one_hot_map)
test_ds_onehot  = test_ds.map(one_hot_map)

In [ ]:
# ============================
# 3️⃣ Baseline Model (Frozen)
# ============================
base_model = EfficientNetB2(
    include_top=False,
    weights="imagenet",
    input_shape=(260, 260, 3)
)
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.5),
    layers.Dense(256, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=["accuracy"]
)

print("\n Starting baseline training...")
history = model.fit(train_ds_onehot, validation_data=val_ds_onehot, epochs=12)

31790344/31790344 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step

 Starting baseline training...
Epoch 1/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 137s 763ms/step - accuracy: 0.1617 - loss: 3.5515 - val_accuracy: 0.4440 - val_loss: 2.5860
Epoch 2/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 53s 413ms/step - accuracy: 0.3819 - loss: 2.7101 - val_accuracy: 0.5172 - val_loss: 2.3836
Epoch 3/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 54s 412ms/step - accuracy: 0.4481 - loss: 2.5568 - val_accuracy: 0.5496 - val_loss: 2.3017
Epoch 4/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 55s 420ms/step - accuracy: 0.4640 - loss: 2.4487 - val_accuracy: 0.5733 - val_loss: 2.2176
Epoch 5/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 53s 413ms/step - accuracy: 0.4834 - loss: 2.3918 - val_accuracy: 0.5733 - val_loss: 2.1771
Epoch 6/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 55s 419ms/step - accuracy: 0.5022 - loss: 2.3517 - val_accuracy: 0.5819 - val_loss: 2.1543
Epoch 7/12
118/118 ━━━━━━━━━━━━━━━━━━━━ 54s 418ms/step - accuracy: 0.5494 - loss: 2.2497 - val_accuracy: 0.5797 - val_loss: 2.1294

In [ ]:
# ============================
# 4️⃣ Random Fine-Tuning Search
# ============================
param_dist = {
    "learning_rate": [1e-4, 5e-5, 1e-5],
    "dropout_dense": [0.3, 0.4, 0.5],
    "dropout_final": [0.3, 0.4, 0.5],
    "dense_units": [128, 256, 512],
    "weight_decay": [1e-3, 1e-4, 1e-5],
}

n_iter = 3
best_val_acc = 0
best_params = None
best_model = None

for i in range(n_iter):
    print(f"\n Fine-tuning Trial {i+1}/{n_iter}")
    params = {k: random.choice(v) for k, v in param_dist.items()}
    print("Params:", params)

    base_model = EfficientNetB2(
        include_top=False,
        weights="imagenet",
        input_shape=(260, 260, 3)
    )
    base_model.trainable = True

    # Only fine-tune the last ~100 layers
    for layer in base_model.layers[:-100]:
        layer.trainable = False

    candidate = models.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dropout(params["dropout_dense"]),
        layers.Dense(params["dense_units"], activation="relu",
                     kernel_regularizer=regularizers.l2(params["weight_decay"])),
        layers.Dropout(params["dropout_final"]),
        layers.Dense(num_classes, activation="softmax")
    ])

    candidate.compile(
        optimizer=tf.keras.optimizers.Adam(params["learning_rate"]),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"]
    )

    history_candidate = candidate.fit(
        train_ds_onehot,
        validation_data=val_ds_onehot,
        epochs=8,
        verbose=1
    )

    val_acc = max(history_candidate.history["val_accuracy"])
    print(f"Validation Accuracy: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_params = params
        best_model = candidate

print("\n Best Params:", best_params)
print(f" Best Validation Accuracy: {best_val_acc:.4f}")


 Fine-tuning Trial 1/3
Params: {'learning_rate': 0.0001, 'dropout_dense': 0.5, 'dropout_final': 0.5, 'dense_units': 256, 'weight_decay': 0.001}
Epoch 1/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 140s 704ms/step - accuracy: 0.0680 - loss: 3.8174 - val_accuracy: 0.2522 - val_loss: 3.3400
Epoch 2/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 61s 469ms/step - accuracy: 0.1637 - loss: 3.4380 - val_accuracy: 0.3815 - val_loss: 2.8823
Epoch 3/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 54s 412ms/step - accuracy: 0.3110 - loss: 3.0167 - val_accuracy: 0.5043 - val_loss: 2.5179
Epoch 4/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 55s 420ms/step - accuracy: 0.4132 - loss: 2.7158 - val_accuracy: 0.5409 - val_loss: 2.3115
Epoch 5/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 82s 419ms/step - accuracy: 0.4815 - loss: 2.4704 - val_accuracy: 0.6164 - val_loss: 2.1399
Epoch 6/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 81s 423ms/step - accuracy: 0.5442 - loss: 2.2881 - val_accuracy: 0.6272 - val_loss: 2.0496
Epoch 7/8
118/118 ━━━━━━━━━━━━━━━━━━━━ 61s 473ms/step - accuracy: 0.5785 -

In [ ]:
# ============================
# 5️⃣ Final Fine-Tuning
# ============================
print("\n Starting final fine-tuning...")
history_finetune = best_model.fit(
    train_ds_onehot,
    validation_data=val_ds_onehot,
    epochs=30
)


 Starting final fine-tuning...
Epoch 1/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 55s 418ms/step - accuracy: 0.6727 - loss: 1.9653 - val_accuracy: 0.6940 - val_loss: 1.8517
Epoch 2/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 54s 422ms/step - accuracy: 0.6806 - loss: 1.9648 - val_accuracy: 0.7047 - val_loss: 1.8438
Epoch 3/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 54s 418ms/step - accuracy: 0.7324 - loss: 1.8276 - val_accuracy: 0.7371 - val_loss: 1.7811
Epoch 4/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 81s 420ms/step - accuracy: 0.7600 - loss: 1.7676 - val_accuracy: 0.7629 - val_loss: 1.7525
Epoch 5/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 61s 468ms/step - accuracy: 0.7751 - loss: 1.7033 - val_accuracy: 0.7500 - val_loss: 1.7600
Epoch 6/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 75s 420ms/step - accuracy: 0.7799 - loss: 1.7106 - val_accuracy: 0.7371 - val_loss: 1.7339
Epoch 7/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 54s 415ms/step - accuracy: 0.8202 - loss: 1.6246 - val_accuracy: 0.7522 - val_loss: 1.7175
Epoch 8/30
118/118 ━━━━━━━━━━━━━━━━━━━━ 59s 459ms/s

In [ ]:
# ============================
# 6️⃣ Temperature Scaling & Metrics
# ============================
TEMPERATURE = 2.0

def predict_with_temperature(model, dataset, T=1.0):
    probs_list, labels_list = [], []
    for imgs, labels in dataset:
        logits = model(imgs, training=False) / T
        probs = tf.nn.softmax(logits).numpy()
        probs_list.extend(probs)
        labels_list.extend(labels.numpy())
    return np.array(labels_list), np.array(probs_list)

y_true, y_probs = predict_with_temperature(best_model, test_ds_onehot, T=TEMPERATURE)

y_pred = np.argmax(y_probs, axis=1)
y_true_int = np.argmax(y_true, axis=1)

present_classes = np.unique(y_true_int)
if len(present_classes) > 1:
    y_true_bin_present = label_binarize(y_true_int, classes=present_classes)
    y_probs_present = y_probs[:, present_classes]
    auc = roc_auc_score(
        y_true_bin_present,
        y_probs_present,
        average="macro",
        multi_class="ovr"
    )
else:
    print("⚠️ Only one class present in test set, AUC not defined.")
    auc = np.nan

accuracy = accuracy_score(y_true_int, y_pred)
precision = precision_score(y_true_int, y_pred, average="weighted", zero_division=0)
recall = recall_score(y_true_int, y_pred, average="weighted", zero_division=0)
f1 = f1_score(y_true_int, y_pred, average="weighted", zero_division=0)
kappa = cohen_kappa_score(y_true_int, y_pred)

print("\n Evaluation Metrics — EfficientNetB2 (260×260)")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"AUC      : {auc:.4f}")
print(f"Kappa    : {kappa:.4f}")


 Evaluation Metrics — EfficientNetB2 (260×260)
Accuracy : 0.5847
Precision: 0.6136
Recall   : 0.5847
F1 Score : 0.5733
AUC      : 0.9073
Kappa    : 0.5692


In [ ]:
import os

def get_model_size(model, model_name="model"):
    temp_h5 = f"{model_name}.h5"
    model.save(temp_h5)
    size_mb = os.path.getsize(temp_h5) / (1024 * 1024)
    print(f" {model_name} Size: {size_mb:.2f} MB")
    os.remove(temp_h5)
    return size_mb

In [ ]:
import os

def get_model_size(model, model_name="best_model"):
    temp_file = f"{model_name}.h5"
    model.save(temp_file)
    size_mb = os.path.getsize(temp_file) / (1024 * 1024)
    print(f"📦 Model Size of {model_name}: {size_mb:.2f} MB")
    os.remove(temp_file)
    return size_mb

# Call for EfficientNetB2 trained model
get_model_size(best_model, "EfficientNetB2_best_model")

📦 Model Size of EfficientNetB2_best_model: 81.69 MB


81.69125366210938